# Module 4.3 — CNN Architectures: LeNet to ResNet

**Module 4: CNNs, Pretrained Models & Deep Dream | Week 1 | Lecture 3**

---

You now know the building blocks: conv layers, pooling, ReLU, fully-connected. The history of CNNs is about how people arranged these blocks — and the ideas that emerged when things stopped working.

This is a conceptual session. We're not implementing these architectures — just understanding what changed at each step and why it mattered.

## Glossary

Today is a tour of historical CNN architectures. Most of these names are landmarks; you'll see them cited in papers and blog posts.

- **LeNet** (1998) — the first practical CNN, designed by Yann LeCun for digit recognition.
- **AlexNet** (2012) — the network that won ImageNet 2012 and kicked off the deep learning revival.
- **VGG** (2014) — a deeper, simpler network: stacks of 3×3 convolutions and pooling, nothing fancy.
- **ResNet** (2015) — introduced *skip connections* to train much deeper networks (50, 100, 152 layers).
- **Skip connection** — a shortcut that bypasses one or more layers, letting the gradient flow straight through. The core trick of ResNet.
- **Residual block** — the building block of ResNet: a small stack of layers plus a skip connection from input to output.
- **Batch normalization** — a layer that normalizes activations within a mini-batch. Speeds up training and helps with stability. (We don't go deep on it in this course; treat it as a black box that helps.)
- **ImageNet** — the dataset (~1.2 million images, 1000 classes) every classical CNN was trained on. Benchmarks the field used for years.
- **Global average pooling** — instead of flattening the final feature map into a long vector, take the average of each channel across all spatial locations. Far fewer parameters than a fully-connected output layer.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

---

## The Timeline

In [ ]:
# Timeline of CNN architectures
fig, ax = plt.subplots(figsize=(15, 5))
ax.set_xlim(1996, 2020)
ax.set_ylim(0, 6)
ax.set_xlabel('Year', fontsize=12)
ax.set_yticks([])
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)

milestones = [
    (1998, 3.5, 'LeNet-5', '60K params\nHandwritten digits', '#4A90D9'),
    (2012, 4.5, 'AlexNet', '60M params\nImageNet winner', '#D9534F'),
    (2014, 3.0, 'VGG-16', '138M params\nSimple & deep', '#5CB85C'),
    (2014.7, 5.0, 'GoogLeNet', '6.8M params\nInception modules', '#F0AD4E'),
    (2015.5, 3.8, 'ResNet', '25M params\nSkip connections', '#9B59B6'),
]

for year, y, name, desc, color in milestones:
    box = mpatches.FancyBboxPatch(
        (year - 0.8, y - 0.6), 1.6, 1.2, boxstyle='round,pad=0.1',
        facecolor=color, edgecolor='white', linewidth=2, alpha=0.9
    )
    ax.add_patch(box)
    ax.text(year, y, name, ha='center', va='center',
            fontsize=10, fontweight='bold', color='white')
    ax.text(year, y - 1.2, desc, ha='center', va='center',
            fontsize=8, color=color)
    ax.plot([year, year], [0.2, y - 0.6], color=color, linewidth=2, alpha=0.5)
    ax.plot(year, 0.2, 'o', color=color, markersize=8)

# Timeline arrow
ax.annotate('', xy=(2019, 0.2), xytext=(1997, 0.2),
            arrowprops=dict(arrowstyle='-|>', color='#333', lw=2))

fig.suptitle('The Evolution of CNN Architectures',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

## LeNet-5 (1998) — Where It Started

**Yann LeCun's** network for reading handwritten zip codes. The template that all CNNs follow.

Architecture: Conv → Pool → Conv → Pool → FC → FC → Output

- 5×5 filters, 2 conv layers
- ~60,000 parameters
- Trained on MNIST-like data

**The big idea:** Convolution + pooling works for visual recognition. This was a radical claim in 1998.

The structure we built last session is essentially a modernized LeNet.

In [ ]:
import torch.nn as nn

# LeNet-5 (modernized slightly with ReLU instead of tanh)
lenet = nn.Sequential(
    nn.Conv2d(1, 6, kernel_size=5),      # 28→24
    nn.ReLU(),
    nn.MaxPool2d(2),                       # 24→12
    nn.Conv2d(6, 16, kernel_size=5),       # 12→8
    nn.ReLU(),
    nn.MaxPool2d(2),                       # 8→4
    nn.Flatten(),
    nn.Linear(16 * 4 * 4, 120),
    nn.ReLU(),
    nn.Linear(120, 84),
    nn.ReLU(),
    nn.Linear(84, 10),
)

params = sum(p.numel() for p in lenet.parameters())
print(f"LeNet-5 parameters: {params:,}")

In [ ]:
# LeNet-5 architecture diagram
fig, ax = plt.subplots(figsize=(14, 3.5))
ax.set_xlim(0, 14)
ax.set_ylim(0, 3.5)
ax.axis('off')

lenet_layers = [
    (1.0,  'Input\n1×28×28',     '#999',    1.4),
    (2.8,  'Conv 5×5\n6×24×24',  '#4A90D9', 1.5),
    (4.4,  'Pool\n6×12×12',      '#F0AD4E', 1.2),
    (5.9,  'Conv 5×5\n16×8×8',   '#4A90D9', 1.5),
    (7.5,  'Pool\n16×4×4',       '#F0AD4E', 1.2),
    (9.1,  'FC\n120',            '#5CB85C', 1.0),
    (10.4, 'FC\n84',             '#5CB85C', 1.0),
    (11.7, 'FC\n10',             '#D9534F', 1.0),
]

for x, label, color, w in lenet_layers:
    box = mpatches.FancyBboxPatch(
        (x - w/2, 0.8), w, 1.6, boxstyle='round,pad=0.1',
        facecolor=color, edgecolor='white', linewidth=2, alpha=0.9
    )
    ax.add_patch(box)
    ax.text(x, 1.6, label, ha='center', va='center',
            fontsize=8, fontweight='bold', color='white')

# Arrows
for i in range(len(lenet_layers) - 1):
    x1 = lenet_layers[i][0] + lenet_layers[i][3] / 2
    x2 = lenet_layers[i+1][0] - lenet_layers[i+1][3] / 2
    ax.annotate('', xy=(x2, 1.6), xytext=(x1, 1.6),
                arrowprops=dict(arrowstyle='-|>', color='#333', lw=1.2))

fig.suptitle('LeNet-5 (1998) — 60K parameters',
             fontsize=13, fontweight='bold', color='#4A90D9')
plt.tight_layout()
plt.show()

---

## AlexNet (2012) — The Deep Learning Revolution

For 14 years after LeNet, CNNs were mostly ignored. Then **Alex Krizhevsky** won the ImageNet competition by a huge margin.

What changed:
- **Much deeper** (8 layers vs LeNet's 2 conv layers)
- **Trained on GPUs** — first major use of GPU for deep learning
- **ReLU activation** instead of sigmoid/tanh
- **Dropout** for regularization
- **ImageNet** — 1.2 million images, 1000 classes (real photos, not just digits)

**The big idea:** With enough data and compute, deeper CNNs massively outperform everything else.

This paper reignited the field.

---

## VGG (2014) — Simple and Deep

The VGG team from Oxford asked: what if we just make it deeper with a simple, uniform architecture?

- **All filters are 3×3** (smaller than AlexNet's 11×11 and 5×5)
- **16 or 19 layers deep**
- Clean, repeating pattern: Conv-Conv-Pool, Conv-Conv-Pool, ...

**The big idea:** Two stacked 3×3 conv layers have the same "receptive field" as one 5×5 layer, but with fewer parameters and more nonlinearity. Smaller filters + more layers = better.

VGG-16 is still widely used today as a feature extractor (we'll use it for Deep Dream).

In [ ]:
# Receptive field comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# One 5×5 filter
ax = axes[0]
ax.set_xlim(-0.5, 4.5)
ax.set_ylim(-0.5, 4.5)
for i in range(5):
    for j in range(5):
        rect = plt.Rectangle((j-0.5, 4-i-0.5), 1, 1,
                              facecolor='#4A90D9', edgecolor='white', alpha=0.7)
        ax.add_patch(rect)
ax.set_title(f'One 5×5 filter\n25 parameters', fontweight='bold')
ax.set_xticks([])
ax.set_yticks([])
ax.set_aspect('equal')

# Two stacked 3×3 filters
ax = axes[1]
ax.set_xlim(-0.5, 4.5)
ax.set_ylim(-0.5, 4.5)
# Outer receptive field (5×5)
for i in range(5):
    for j in range(5):
        rect = plt.Rectangle((j-0.5, 4-i-0.5), 1, 1,
                              facecolor='#5CB85C', edgecolor='white', alpha=0.3)
        ax.add_patch(rect)
# Inner 3×3 (second layer)
for i in range(3):
    for j in range(3):
        rect = plt.Rectangle((j+0.5, 3-i-0.5), 1, 1,
                              facecolor='#5CB85C', edgecolor='white', alpha=0.7)
        ax.add_patch(rect)
ax.set_title(f'Two stacked 3×3 filters\n18 parameters + 2 ReLUs', fontweight='bold')
ax.set_xticks([])
ax.set_yticks([])
ax.set_aspect('equal')

plt.suptitle('Same Receptive Field, Fewer Parameters, More Nonlinearity',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Two 3×3 layers: 2 × (3×3) = 18 parameters + 2 ReLU activations")
print("One 5×5 layer:  1 × (5×5) = 25 parameters + 1 ReLU activation")
print("\nFewer parameters AND more nonlinearity → better features.")

In [ ]:
# VGG-16 architecture diagram — shows the repeating block pattern
fig, ax = plt.subplots(figsize=(15, 4.5))
ax.set_xlim(0, 15)
ax.set_ylim(0, 4.5)
ax.axis('off')

# VGG-16 blocks: (x, label, color, width, height_scale)
# Group by blocks to show the repeating pattern
blocks = [
    # Block 1: 2 convs + pool
    [(0.8, 'C 64', '#4A90D9'), (1.6, 'C 64', '#4A90D9'), (2.3, 'P', '#F0AD4E')],
    # Block 2: 2 convs + pool
    [(3.1, 'C 128', '#4A90D9'), (3.9, 'C 128', '#4A90D9'), (4.6, 'P', '#F0AD4E')],
    # Block 3: 3 convs + pool
    [(5.4, 'C 256', '#4A90D9'), (6.1, 'C 256', '#4A90D9'), (6.8, 'C 256', '#4A90D9'), (7.4, 'P', '#F0AD4E')],
    # Block 4: 3 convs + pool
    [(8.1, 'C 512', '#4A90D9'), (8.8, 'C 512', '#4A90D9'), (9.5, 'C 512', '#4A90D9'), (10.1, 'P', '#F0AD4E')],
    # Block 5: 3 convs + pool
    [(10.8, 'C 512', '#4A90D9'), (11.5, 'C 512', '#4A90D9'), (12.2, 'C 512', '#4A90D9'), (12.8, 'P', '#F0AD4E')],
]

# FC layers
fc_layers = [(13.5, 'FC\n4096', '#5CB85C'), (14.2, 'FC\n1K', '#D9534F')]

# Draw blocks
spatial = ['224', '224', '', '112', '112', '', '56', '56', '56', '',
           '28', '28', '28', '', '14', '14', '14', '', '7', '7']

all_layers = []
for block in blocks:
    all_layers.extend(block)

for x, label, color in all_layers:
    is_pool = label == 'P'
    w = 0.45 if is_pool else 0.6
    h = 1.2 if is_pool else 2.0
    y_off = 1.2 if is_pool else 0.8

    box = mpatches.FancyBboxPatch(
        (x - w/2, y_off), w, h, boxstyle='round,pad=0.05',
        facecolor=color, edgecolor='white', linewidth=1.5, alpha=0.9
    )
    ax.add_patch(box)
    ax.text(x, y_off + h/2, label, ha='center', va='center',
            fontsize=7, fontweight='bold', color='white', rotation=90 if not is_pool else 0)

for x, label, color in fc_layers:
    box = mpatches.FancyBboxPatch(
        (x - 0.3, 1.0), 0.6, 1.6, boxstyle='round,pad=0.05',
        facecolor=color, edgecolor='white', linewidth=1.5, alpha=0.9
    )
    ax.add_patch(box)
    ax.text(x, 1.8, label, ha='center', va='center',
            fontsize=7, fontweight='bold', color='white')

# Block labels
block_centers = [1.2, 3.5, 6.25, 9.1, 11.8]
block_names = ['Block 1\n224×224', 'Block 2\n112×112', 'Block 3\n56×56',
               'Block 4\n28×28', 'Block 5\n14×14']
for x, name in zip(block_centers, block_names):
    ax.text(x, 0.3, name, ha='center', fontsize=7, color='#666', fontstyle='italic')

ax.text(13.85, 0.3, 'Classifier', ha='center', fontsize=7, color='#666', fontstyle='italic')

# Bracket showing "all 3×3"
ax.text(7.5, 3.5, 'All conv layers use 3×3 filters — the VGG principle',
        ha='center', fontsize=10, fontweight='bold', color='#5CB85C')

fig.suptitle('VGG-16 (2014) — 138M parameters, 13 conv + 3 FC layers',
             fontsize=13, fontweight='bold', color='#5CB85C')
plt.tight_layout()
plt.show()

---

## ResNet (2015) — The Skip Connection

As networks got deeper (20, 50, 100+ layers), a strange problem appeared: **deeper networks performed worse** than shallower ones, even on training data. This wasn't overfitting — it was a training problem.

The gradient signal was getting lost as it traveled through dozens of layers. By the time it reached the early layers, there was nothing useful left to learn from.

**ResNet's solution: skip connections** (also called residual connections).

Instead of learning `output = F(input)`, each block learns `output = F(input) + input`.

The `+ input` is a shortcut that lets the gradient flow directly backward through the network, skipping layers. If a layer has nothing useful to add, it can learn to output zero and just pass the input through.

In [ ]:
# Skip connection diagram
fig, ax = plt.subplots(figsize=(12, 4))
ax.set_xlim(0, 12)
ax.set_ylim(0, 4)
ax.axis('off')

# Standard block (no skip)
for x, label, color in [(1.5, 'Conv', '#4A90D9'), (3.5, 'ReLU', '#F0AD4E'),
                          (5.5, 'Conv', '#4A90D9')]:
    box = mpatches.FancyBboxPatch(
        (x - 0.7, 1.0), 1.4, 1.0, boxstyle='round,pad=0.1',
        facecolor=color, edgecolor='white', linewidth=2
    )
    ax.add_patch(box)
    ax.text(x, 1.5, label, ha='center', va='center',
            fontsize=10, fontweight='bold', color='white')

# Arrows through the block
for s, e in [(0.3, 0.8), (2.2, 2.8), (4.2, 4.8), (6.2, 6.8)]:
    ax.annotate('', xy=(e, 1.5), xytext=(s, 1.5),
                arrowprops=dict(arrowstyle='-|>', color='#333', lw=1.5))

# Skip connection (curved arrow over the top)
ax.annotate('', xy=(6.8, 1.5), xytext=(0.3, 1.5),
            arrowprops=dict(arrowstyle='-|>', color='#D9534F', lw=2.5,
                           connectionstyle='arc3,rad=-0.4'))
ax.text(3.5, 3.3, 'Skip Connection (+ input)', ha='center',
        fontsize=11, fontweight='bold', color='#D9534F')

# Plus sign
circle = plt.Circle((7.2, 1.5), 0.3, facecolor='#D9534F',
                     edgecolor='white', linewidth=2)
ax.add_patch(circle)
ax.text(7.2, 1.5, '+', ha='center', va='center',
        fontsize=16, fontweight='bold', color='white')

# ReLU after add
box = mpatches.FancyBboxPatch(
    (8.3, 1.0), 1.4, 1.0, boxstyle='round,pad=0.1',
    facecolor='#F0AD4E', edgecolor='white', linewidth=2
)
ax.add_patch(box)
ax.text(9.0, 1.5, 'ReLU', ha='center', va='center',
        fontsize=10, fontweight='bold', color='white')

ax.annotate('', xy=(8.3, 1.5), xytext=(7.5, 1.5),
            arrowprops=dict(arrowstyle='-|>', color='#333', lw=1.5))
ax.annotate('', xy=(10.2, 1.5), xytext=(9.7, 1.5),
            arrowprops=dict(arrowstyle='-|>', color='#333', lw=1.5))

ax.text(0.3, 1.5, 'x', fontsize=12, fontweight='bold', ha='center')
ax.text(10.5, 1.5, 'output', fontsize=10, fontweight='bold')
ax.text(3.5, 0.3, 'output = F(x) + x', ha='center',
        fontsize=11, color='#666', fontstyle='italic')

fig.suptitle('The Residual Block (ResNet)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**The big idea:** Skip connections let you train networks that are 50, 100, even 152 layers deep. The gradient has a direct highway to travel backward. This unlocked "really deep" learning.

ResNet won ImageNet 2015 with 152 layers and a 3.6% error rate — surpassing human performance for the first time.

In [ ]:
# Side-by-side architecture comparison: LeNet vs VGG-16 vs ResNet-50
# Drawn as stacked vertical blocks — taller = more layers, wider = more channels
fig, axes = plt.subplots(1, 3, figsize=(15, 7))

def draw_arch(ax, title, color, layers, params, year):
    """Draw an architecture as stacked horizontal bars.
    layers: list of (label, width_fraction, bar_color)
    """
    ax.set_xlim(0, 10)
    ax.set_ylim(-0.5, len(layers) + 0.5)
    ax.set_yticks([])
    ax.set_xticks([])
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.spines['bottom'].set_visible(False)

    for i, (label, width_frac, bar_color) in enumerate(layers):
        y = len(layers) - 1 - i
        w = width_frac * 8
        x_start = (10 - w) / 2
        bar = mpatches.FancyBboxPatch(
            (x_start, y + 0.1), w, 0.7, boxstyle='round,pad=0.05',
            facecolor=bar_color, edgecolor='white', linewidth=1.5, alpha=0.9
        )
        ax.add_patch(bar)
        ax.text(5, y + 0.45, label, ha='center', va='center',
                fontsize=7.5, fontweight='bold', color='white')

    ax.set_title(f'{title}\n{year} · {params}', fontsize=12,
                 fontweight='bold', color=color, pad=10)

# LeNet-5
lenet_layers = [
    ('Input: 1×28×28',       0.3,  '#999'),
    ('Conv 5×5 → 6×24×24',   0.35, '#4A90D9'),
    ('MaxPool → 6×12×12',    0.3,  '#F0AD4E'),
    ('Conv 5×5 → 16×8×8',    0.4,  '#4A90D9'),
    ('MaxPool → 16×4×4',     0.3,  '#F0AD4E'),
    ('FC 256→120',            0.55, '#5CB85C'),
    ('FC 120→84',             0.5,  '#5CB85C'),
    ('FC 84→10',              0.4,  '#D9534F'),
]
draw_arch(axes[0], 'LeNet-5', '#4A90D9', lenet_layers, '60K params', '1998')

# VGG-16 (abbreviated)
vgg_layers = [
    ('Input: 3×224×224',       0.3,  '#999'),
    ('2× Conv 3×3 → 64',      0.45, '#4A90D9'),
    ('Pool → 112×112',         0.3,  '#F0AD4E'),
    ('2× Conv 3×3 → 128',     0.55, '#4A90D9'),
    ('Pool → 56×56',           0.3,  '#F0AD4E'),
    ('3× Conv 3×3 → 256',     0.65, '#4A90D9'),
    ('Pool → 28×28',           0.3,  '#F0AD4E'),
    ('3× Conv 3×3 → 512',     0.8,  '#4A90D9'),
    ('Pool → 14×14',           0.3,  '#F0AD4E'),
    ('3× Conv 3×3 → 512',     0.8,  '#4A90D9'),
    ('Pool → 7×7',             0.3,  '#F0AD4E'),
    ('FC 25088→4096',          1.0,  '#5CB85C'),
    ('FC 4096→4096',           1.0,  '#5CB85C'),
    ('FC 4096→1000',           0.6,  '#D9534F'),
]
draw_arch(axes[1], 'VGG-16', '#5CB85C', vgg_layers, '138M params', '2014')

# ResNet-50 (abbreviated)
resnet_layers = [
    ('Input: 3×224×224',       0.3,  '#999'),
    ('Conv 7×7 → 64, stride 2', 0.45, '#4A90D9'),
    ('MaxPool → 56×56',       0.3,  '#F0AD4E'),
    ('3× Residual Block (64)',  0.5,  '#9B59B6'),
    ('4× Residual Block (128)', 0.6,  '#9B59B6'),
    ('6× Residual Block (256)', 0.7,  '#9B59B6'),
    ('3× Residual Block (512)', 0.8,  '#9B59B6'),
    ('Global AvgPool → 512',   0.35, '#F0AD4E'),
    ('FC 512→1000',            0.5,  '#D9534F'),
]
draw_arch(axes[2], 'ResNet-50', '#9B59B6', resnet_layers, '25M params', '2015')

plt.suptitle('Architecture Comparison: Layer Structure Side by Side',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("Key observations:")
print("• LeNet is tiny — 8 layers, straightforward")
print("• VGG-16 is deep but simple — same pattern repeated, huge FC layers at the end")
print("• ResNet-50 packs 50 layers into residual blocks — global avg pool instead of huge FC layers")
print("• VGG's FC layers (4096×2) account for most of its 138M params")
print("• ResNet avoids this with global average pooling → single FC layer")

---

## Comparison Table

In [ ]:
# Architecture comparison
data = {
    'Architecture': ['LeNet-5', 'AlexNet', 'VGG-16', 'ResNet-50'],
    'Year': [1998, 2012, 2014, 2015],
    'Layers': [5, 8, 16, 50],
    'Parameters': ['60K', '60M', '138M', '25M'],
    'Key Innovation': [
        'Conv + Pool works',
        'Deep + GPU + ReLU',
        'All 3×3, go deeper',
        'Skip connections',
    ],
    'ImageNet Top-5 Error': ['—', '16.4%', '7.3%', '3.6%'],
}

# Print as table
header = f"{'Architecture':<14} {'Year':>5} {'Layers':>7} {'Params':>10} {'Key Innovation':<25} {'Top-5 Error':>12}"
print(header)
print('=' * len(header))
for i in range(4):
    print(f"{data['Architecture'][i]:<14} {data['Year'][i]:>5} {data['Layers'][i]:>7} "
          f"{data['Parameters'][i]:>10} {data['Key Innovation'][i]:<25} {data['ImageNet Top-5 Error'][i]:>12}")

In [ ]:
# Depth vs error rate visualization
years = [1998, 2012, 2014, 2015]
depths = [5, 8, 16, 50]
errors = [None, 16.4, 7.3, 3.6]
names = ['LeNet-5', 'AlexNet', 'VGG-16', 'ResNet-50']
colors = ['#4A90D9', '#D9534F', '#5CB85C', '#9B59B6']

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Depth over time
axes[0].bar(range(4), depths, color=colors, edgecolor='white', linewidth=2)
axes[0].set_xticks(range(4))
axes[0].set_xticklabels([f"{n}\n({y})" for n, y in zip(names, years)])
axes[0].set_ylabel('Number of Layers')
axes[0].set_title('Networks Got Deeper', fontweight='bold')
for i, d in enumerate(depths):
    axes[0].text(i, d + 1, str(d), ha='center', fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')

# Error rate over time (skip LeNet, no ImageNet number)
axes[1].plot([2012, 2014, 2015], [16.4, 7.3, 3.6], 'o-',
            color='#D9534F', linewidth=2.5, markersize=10)
axes[1].axhline(y=5.1, color='#999', linestyle='--', linewidth=1.5, label='Human (~5.1%)')
for y, e, n in [(2012, 16.4, 'AlexNet'), (2014, 7.3, 'VGG'), (2015, 3.6, 'ResNet')]:
    axes[1].annotate(f"{n}\n{e}%", (y, e), textcoords='offset points',
                     xytext=(15, 5), fontsize=9, fontweight='bold')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('ImageNet Top-5 Error (%)')
axes[1].set_title('Error Rates Plummeted', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, 20)

plt.suptitle('The CNN Revolution', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In just three years (2012-2015), ImageNet error dropped from 16.4% to 3.6% — below human performance. The key ingredients: more layers, better architectures, and skip connections.

---

## What You Need to Remember

You don't need to memorize architecture details. Here's what matters:

1. **CNNs work because of convolution** — local filters, parameter sharing, translation invariance
2. **Deeper is generally better** — more layers learn more abstract features
3. **3×3 filters won** — they're the standard (VGG proved this)
4. **Skip connections enable very deep networks** — ResNet solved the training problem
5. **You don't train these from scratch** — you use pretrained models (Week 2's topic)

When you load a pretrained ResNet next week, you'll know what's inside it and why it was designed that way.

---

## Exercises

### Exercise 1: Architecture reasoning

Why does ResNet have *fewer* parameters than VGG (25M vs 138M) despite being much deeper (50 vs 16 layers)? What design choices account for the difference?

### Exercise 2: Skip connection intuition

In a ResNet block, `output = F(x) + x`. If the block learns F(x) = 0 (all zeros), what does the block do? Why is this useful?

### Exercise 3: Design your own

If you were designing a CNN for 64×64 color images with 20 classes, what would your architecture look like? How many conv layers? What filter sizes? When would you pool? Sketch it out (no code needed, just the layer sequence with dimensions).

In [ ]:
# Exercises: Your answers here


---

## Summary

| Architecture | Year | Key Idea |
|:---|:---|:---|
| **LeNet-5** | 1998 | Conv + Pool works for vision |
| **AlexNet** | 2012 | Deep + GPU + ReLU = breakthrough |
| **VGG** | 2014 | Keep it simple: all 3×3, just go deeper |
| **ResNet** | 2015 | Skip connections let you train 100+ layers |

The trend: deeper networks with clever training tricks. Each breakthrough reduced ImageNet error and changed what was considered possible.

---

**Next session (Lab):** Apply hand-crafted filters to real images, then let PyTorch learn its own filters.